# Data Quality via Great Expectations

**Great Expectations (GX)** describes data quality as **expectations**: declarative, human-readable statements such as "values in `fare_amount` are between 0 and 500" or "`trip_id` is unique". A **suite** groups expectations, a **validation definition** binds a suite to a batch of data, and a **checkpoint** runs validations and triggers actions. Results are structured, so pipelines can stop on failure and teams can publish data quality reports.

**After this lesson you can:**

1. Create a GX data context, data source, asset and batch definition.
2. Write expectation suites for completeness, validity, uniqueness and distribution.
3. Validate batches, read the results and decide pass/fail in a pipeline.
4. Compare GX with dbt tests and pandera, and choose a tool.

**Data:** NYC taxi trips (January 2024 as the reference batch, February 2024 as the new delivery).

In [1]:
import contextlib
import io
import sys
import tempfile
import time
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
WORK = Path(tempfile.mkdtemp())

@contextlib.contextmanager
def quiet():
    """Collect the verbose log output of a tool (stderr) instead of printing it in the notebook."""
    buf = io.StringIO()
    with contextlib.redirect_stderr(buf):
        yield buf

import os
import warnings

os.environ["GX_ANALYTICS_ENABLED"] = "False"
os.environ["TQDM_DISABLE"] = "1"                   # no progress bars in the notebook output
warnings.filterwarnings("ignore")
import great_expectations as gx

COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "PULocationID", "payment_type",
        "fare_amount", "tip_amount", "total_amount"]
jan = pd.read_parquet(datasets.taxi("2024-01"), columns=COLS)
feb = pd.read_parquet(datasets.taxi("2024-02"), columns=COLS)
print("GX", gx.__version__, "| January:", len(jan), "| February:", len(feb))

GX 1.25.0 | January: 2964624 | February: 3007526


## 1. Context, Data Source, Asset, Batch

| GX object | Meaning |
| :--- | :--- |
| data context | the GX project (here *ephemeral*: in memory) |
| data source | where data lives (pandas, Spark, SQL databases) |
| data asset | a table, file or DataFrame inside the source |
| batch definition | how to select a batch (the whole frame, a month, ...) |

In [2]:
from great_expectations.data_context.types.base import ProgressBarsConfig

ctx = gx.get_context(mode="ephemeral")
ctx.variables.progress_bars = ProgressBarsConfig(globally=False, metric_calculations=False)   # no progress bars in the output
source = ctx.data_sources.add_pandas("taxi")
asset = source.add_dataframe_asset("yellow_trips")
batch_def = asset.add_batch_definition_whole_dataframe("monthly_file")
print(type(ctx).__name__, "|", source.name, "/", asset.name, "/", batch_def.name)

EphemeralDataContext | taxi / yellow_trips / monthly_file


## 2. An Expectation Suite

Expectations cover the main dimensions of data quality:

| Dimension | Expectation |
| :--- | :--- |
| completeness | `ExpectColumnValuesToNotBeNull` (with `mostly=` for tolerated gaps) |
| validity | `ExpectColumnValuesToBeBetween`, `ExpectColumnValuesToBeInSet` |
| consistency | `ExpectColumnPairValuesAToBeGreaterThanB` |
| volume | `ExpectTableRowCountToBeBetween` |
| distribution | `ExpectColumnMeanToBeBetween`, `ExpectColumnQuantileValuesToBeBetween` |
| schema | `ExpectTableColumnsToMatchSet` |

In [3]:
E = gx.expectations
suite = ctx.suites.add(gx.ExpectationSuite(name="yellow_trips_quality"))
for exp in [
    E.ExpectTableColumnsToMatchSet(column_set=COLS),
    E.ExpectTableRowCountToBeBetween(min_value=2_000_000, max_value=4_000_000),
    E.ExpectColumnValuesToNotBeNull(column="tpep_pickup_datetime"),
    E.ExpectColumnValuesToNotBeNull(column="passenger_count", mostly=0.95),
    E.ExpectColumnValuesToBeBetween(column="fare_amount", min_value=0, max_value=500, mostly=0.98),   # ~1.3 % reversals in January
    E.ExpectColumnValuesToBeBetween(column="trip_distance", min_value=0, max_value=100, mostly=0.999),
    E.ExpectColumnValuesToBeInSet(column="payment_type", value_set=[0, 1, 2, 3, 4, 5, 6]),
    E.ExpectColumnValuesToBeBetween(column="PULocationID", min_value=1, max_value=265),
    E.ExpectColumnPairValuesAToBeGreaterThanB(column_A="tpep_dropoff_datetime", column_B="tpep_pickup_datetime",
                                             or_equal=True, mostly=0.99),
    E.ExpectColumnMeanToBeBetween(column="fare_amount", min_value=15, max_value=22),
    E.ExpectColumnQuantileValuesToBeBetween(column="trip_distance", quantile_ranges={
        "quantiles": [0.5, 0.9], "value_ranges": [[1.0, 2.5], [5.0, 15.0]]}),
]:
    suite.add_expectation(exp)
print(f"suite '{suite.name}' with {len(suite.expectations)} expectations")

suite 'yellow_trips_quality' with 11 expectations


## 3. Validate the Reference Batch

In [4]:
validation = ctx.validation_definitions.add(gx.ValidationDefinition(name="validate_month", data=batch_def, suite=suite))

def run(df: pd.DataFrame) -> pd.DataFrame:
    with quiet():
        res = validation.run(batch_parameters={"dataframe": df})
    rows = []
    for r in res.results:
        cfg = r.expectation_config
        rows.append({"expectation": cfg.type, "column": cfg.kwargs.get("column", cfg.kwargs.get("column_A", "")),
                     "success": r.success, "unexpected_%": round(r.result.get("unexpected_percent") or 0, 3),
                     "observed": r.result.get("observed_value")})
    return pd.DataFrame(rows), res.success

jan_report, jan_ok = run(jan)
print("January batch passes:", jan_ok)
jan_report

January batch passes: True


,expectation,column,success,unexpected_%,observed
0,expect_table_columns_to_match_set,,True,0.000,"[PULocationID, fare_amount, passenger_count, p..."
1,expect_table_row_count_to_be_between,,True,0.000,2964624
2,expect_column_pair_values_a_to_be_greater_than_b,tpep_dropoff_datetime,True,0.002,None
3,expect_column_values_to_not_be_null,tpep_pickup_datetime,True,0.000,None
4,expect_column_values_to_not_be_null,passenger_count,True,4.728,None
5,expect_column_values_to_be_between,fare_amount,True,1.265,None
6,expect_column_mean_to_be_between,fare_amount,True,0.000,18.175062
7,expect_column_values_to_be_between,trip_distance,True,0.002,None
8,expect_column_quantile_values_to_be_between,trip_distance,True,0.000,"{'quantiles': ['0.5', '0.9'], 'values': ['1.68..."
9,expect_column_values_to_be_in_set,payment_type,True,0.000,None


## 4. Validate a New Delivery, and a Broken One

February is the "new delivery". A third batch simulates a broken upstream system: fares in **cents** instead of dollars for 30 % of the rows, and the pickup location column filled with zeros for 5 %.

In [5]:
feb_report, feb_ok = run(feb)
broken = feb.copy()
rng = np.random.default_rng(42)
idx = rng.random(len(broken)) < 0.30
broken.loc[idx, "fare_amount"] = broken.loc[idx, "fare_amount"] * 100            # unit change in an upstream system
broken.loc[rng.random(len(broken)) < 0.05, "PULocationID"] = 0
broken_report, broken_ok = run(broken)
compare = jan_report[["expectation", "column", "success"]].rename(columns={"success": "Jan"}).assign(
    Feb=feb_report["success"], broken=broken_report["success"])
print(f"January: {jan_ok} | February: {feb_ok} | broken delivery: {broken_ok}")
compare

January: True | February: False | broken delivery: False


,expectation,column,Jan,Feb,broken
0,expect_table_columns_to_match_set,,True,True,True
1,expect_table_row_count_to_be_between,,True,True,True
2,expect_column_pair_values_a_to_be_greater_than_b,tpep_dropoff_datetime,True,True,True
3,expect_column_values_to_not_be_null,tpep_pickup_datetime,True,True,True
4,expect_column_values_to_not_be_null,passenger_count,True,False,False
5,expect_column_values_to_be_between,fare_amount,True,True,False
6,expect_column_mean_to_be_between,fare_amount,True,True,False
7,expect_column_values_to_be_between,trip_distance,True,True,True
8,expect_column_quantile_values_to_be_between,trip_distance,True,True,True
9,expect_column_values_to_be_in_set,payment_type,True,True,True


In [6]:
failed = broken_report[~broken_report["success"]][["expectation", "column", "unexpected_%", "observed"]]
failed

,expectation,column,unexpected_%,observed
4,expect_column_values_to_not_be_null,passenger_count,6.172,None
5,expect_column_values_to_be_between,fare_amount,30.414,None
6,expect_column_mean_to_be_between,fare_amount,0.000,554.999346
10,expect_column_values_to_be_between,PULocationID,5.000,None


February fails one rule: `passenger_count` is missing in about 6 % of its rows, above the 5 % tolerance (January: about 4.7 %). Is it drift in the source system or a real problem? The validation cannot decide; it makes the change visible, and a person decides whether to fix the source or adjust the tolerance. The broken delivery fails more rules: the validity and distribution expectations that a unit change and a broken column produce. The `mostly` parameter decides how strict each rule is: real data always has a few exceptions (the January batch contains some negative fares), and a pipeline must not stop for those.

## 5. Use Results in a Pipeline: a Quality Gate

In [7]:
def load_if_valid(df: pd.DataFrame, name: str) -> str:
    report, ok = run(df)
    if not ok:
        bad = report.loc[~report["success"], "expectation"].tolist()
        return f"{name}: REJECTED ({len(bad)} failed expectations: {', '.join(bad[:3])}{'...' if len(bad) > 3 else ''})"
    return f"{name}: loaded ({len(df):,} rows)"

for df, name in [(feb, "February delivery"), (broken, "broken delivery")]:
    print(load_if_valid(df, name))

February delivery: REJECTED (1 failed expectations: expect_column_values_to_not_be_null)


broken delivery: REJECTED (4 failed expectations: expect_column_values_to_not_be_null, expect_column_values_to_be_between, expect_column_mean_to_be_between...)


In production, a **checkpoint** wraps validation definitions with **actions** (store results, update Data Docs, send a Slack message), and the orchestrator (Airflow, Dagster) stops downstream tasks when validation fails.

## 6. Choosing a Data Quality Tool

| Tool | Where it fits | Strength |
| :--- | :--- | :--- |
| dbt tests (Lesson 03) | SQL models in the warehouse | tests live next to the transformations |
| Great Expectations | any source: pandas, Spark, SQL | rich expectation library, structured results, Data Docs |
| pandera (M10 L14) | Python/DataFrame code | schemas as code, type-checker friendly |
| Soda, Monte Carlo, Elementary | monitoring over time | anomaly detection on metrics, freshness, volume |

*Insight: Write data quality as declarative expectations with explicit tolerances (`mostly`), validate every delivery before it is loaded, and let failures stop the pipeline and produce a readable report. Unit changes and broken columns are caught by distribution and validity checks long before a dashboard shows strange numbers.*

## Real-World Use Case or Analogy

Expectations are like a **food safety inspection checklist**: temperature within limits, labels present, no more than a few damaged items per crate (`mostly`). A delivery that fails is refused at the door, with a written report for the supplier.

Data teams publish GX Data Docs as a shared "quality report" for each dataset; data contracts between teams (M12) often start as an expectation suite agreed by both sides.